# Phase 1 Environment Debug

This notebook creates one ManiSkill environment, exports RGB/depth/segmentation metadata, and writes an Open3D point cloud.

In [ ]:
from pathlib import Path
import logging
import sys

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(level=logging.INFO, format="%(levelname)s:%(name)s:%(message)s")

In [ ]:
ENV_ID = "PickCube-v1"
OBS_MODE = "rgbd"
CAMERA_NAME = None
SEED = 0
OUTPUT_DIR = Path("outputs/env_debug_notebook")

In [ ]:
from src.env.maniskill_env import ManiSkillScene
from src.io.export_utils import export_observation_frame
from src.geometry.rgbd_to_pointcloud import generate_and_save_pointcloud

with ManiSkillScene(env_name=ENV_ID, obs_mode=OBS_MODE, camera_name=CAMERA_NAME) as scene:
    scene.reset(seed=SEED)
    frame = scene.get_observation(camera_name=CAMERA_NAME)
    print("RGB:", None if frame.rgb is None else (frame.rgb.shape, frame.rgb.dtype))
    print("Depth:", None if frame.depth is None else (frame.depth.shape, frame.depth.dtype))
    print("Segmentation:", None if frame.segmentation is None else (frame.segmentation.shape, frame.segmentation.dtype))
    print("Source keys:", frame.source_keys)
    paths = export_observation_frame(frame, OUTPUT_DIR, env_name=ENV_ID, step_name="reset")
    if frame.rgb is not None and frame.depth is not None:
        paths["pointcloud"] = generate_and_save_pointcloud(
            rgb=frame.rgb,
            depth=frame.depth,
            intrinsic=frame.camera_info.intrinsic,
            extrinsic=frame.camera_info.extrinsic,
            output_path=OUTPUT_DIR / "pointcloud.ply",
        )
    paths